In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/rohan0301/unsupervised-learning-on-country-data/Country-data.csv')
dd = pd.read_csv('/kaggle/input/datasets/rohan0301/unsupervised-learning-on-country-data/data-dictionary.csv')

print("Shape:", df.shape)
print("\nData Dictionary:\n", dd.to_string(index=False))
print("\nFirst 5 rows:\n", df.head())
print("\nData Types:\n", df.dtypes)
print("\nMissing Values:\n", df.isnull().sum())

In [ ]:
print("Statistical Summary:\n")
df.describe().T.style.background_gradient(cmap='YlOrRd')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
print("Libraries loaded successfully!")

In [ ]:

features = df.columns[1:]
plt.figure(figsize=(12, 8))
corr = df[features].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, annot_kws={"size": 10})

plt.title('Feature Correlation Heatmap', fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
key_features = ['child_mort', 'income', 'gdpp', 'life_expec', 'inflation']
sns.pairplot(df[key_features], diag_kind='kde', plot_kws={'alpha': 0.5, 'color': 'steelblue'})
plt.suptitle('Pairplot of Key Features', y=1.02, fontsize=14)
plt.show()

In [ ]:
X = df[features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=features)
print("Scaled Data (first 5 rows):")
X_scaled_df.head()

In [ ]:
inertia = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertia, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (K)', fontsize=12)
axes[0].set_ylabel('Inertia', fontsize=12)
axes[0].set_title('Elbow Method', fontsize=14)
axes[0].axvline(x=3, color='red', linestyle='--', label='Optimal K=3')
axes[0].legend()

axes[1].plot(K_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (K)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score vs K', fontsize=14)
axes[1].axvline(x=3, color='red', linestyle='--', label='Optimal K=3')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nBest K by Silhouette Score: {K_range[np.argmax(silhouette_scores)]} "
      f"(score = {max(silhouette_scores):.4f})")

In [ ]:
optimal_k = 3
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['KMeans_Cluster'] = kmeans.fit_predict(X_scaled)

print("Cluster Distribution:")
print(df['KMeans_Cluster'].value_counts().sort_index())

print("\nCluster-wise Mean Values:")
df.groupby('KMeans_Cluster')[features].mean().round(2)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"Explained Variance Ratio: {pca.explained_variance_ratio_}")
print(f"Total Variance Explained: {sum(pca.explained_variance_ratio_):.2%}")

pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
pca_df['Cluster'] = df['KMeans_Cluster']
pca_df['Country'] = df['country']

plt.figure(figsize=(12, 8))
colors = ['#e74c3c', '#2ecc71', '#3498db']
cluster_names = {0: 'Cluster 0', 1: 'Cluster 1', 2: 'Cluster 2'}

for cluster in range(optimal_k):
    mask = pca_df['Cluster'] == cluster
    plt.scatter(pca_df.loc[mask, 'PC1'], pca_df.loc[mask, 'PC2'],
                c=colors[cluster], label=f'Cluster {cluster}',
                alpha=0.7, s=80, edgecolors='white', linewidth=0.5)

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
plt.title('KMeans Clusters Visualized via PCA', fontsize=14)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 9))

for cluster in range(optimal_k):
    mask = pca_df['Cluster'] == cluster
    plt.scatter(pca_df.loc[mask, 'PC1'], pca_df.loc[mask, 'PC2'],
                c=colors[cluster], label=f'Cluster {cluster}',
                alpha=0.7, s=80, edgecolors='white')

# Annotate extreme countries for storytelling
extremes = pca_df.nlargest(5, 'PC1').index.tolist() + pca_df.nsmallest(5, 'PC1').index.tolist()
for idx in extremes:
    plt.annotate(pca_df.loc[idx, 'Country'],
                 (pca_df.loc[idx, 'PC1'], pca_df.loc[idx, 'PC2']),
                 fontsize=8, alpha=0.85,
                 xytext=(5, 5), textcoords='offset points')

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
plt.title('Country Clusters with Labels (PCA)', fontsize=14)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(18, 7))
linked = linkage(X_scaled, method='ward')

dendrogram(linked,
           labels=df['country'].values,
           leaf_rotation=90,
           leaf_font_size=6,
           color_threshold=10)

plt.title('Hierarchical Clustering Dendrogram (Ward Linkage)', fontsize=14)
plt.xlabel('Countries', fontsize=12)
plt.ylabel('Euclidean Distance', fontsize=12)
plt.axhline(y=10, color='red', linestyle='--', label='Cut threshold')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
agg = AgglomerativeClustering(n_clusters=3, linkage='ward')
df['Agg_Cluster'] = agg.fit_predict(X_scaled)

print("Agglomerative Cluster Distribution:")
print(df['Agg_Cluster'].value_counts().sort_index())

# Silhouette comparison
km_score = silhouette_score(X_scaled, df['KMeans_Cluster'])
agg_score = silhouette_score(X_scaled, df['Agg_Cluster'])

print(f"\nKMeans Silhouette Score:          {km_score:.4f}")
print(f"Agglomerative Silhouette Score:   {agg_score:.4f}")
print(f"\nBetter Model: {'KMeans' if km_score > agg_score else 'Agglomerative'}")

In [ ]:
cluster_profile = df.groupby('KMeans_Cluster')[features].mean().round(2)

# Assign human-readable labels based on socio-economic profile
labels_map = {}
gdpp_order = cluster_profile['gdpp'].rank()
for c in cluster_profile.index:
    if gdpp_order[c] == 1:
        labels_map[c] = 'Underdeveloped'
    elif gdpp_order[c] == 2:
        labels_map[c] = 'Developing'
    else:
        labels_map[c] = 'Developed'

df['Cluster_Label'] = df['KMeans_Cluster'].map(labels_map)

print("=== Cluster Profiles ===\n")
for c in sorted(labels_map):
    print(f"Cluster {c} → {labels_map[c]}")
    print(cluster_profile.loc[c])
    print()

In [ ]:
# Countries in the "Underdeveloped" cluster — highest priority for HELP
needy = df[df['Cluster_Label'] == 'Underdeveloped'][['country', 'child_mort', 'income', 'gdpp', 'life_expec']]
needy = needy.sort_values('child_mort', ascending=False).reset_index(drop=True)

print(f"Countries Needing Most Aid ({len(needy)} total):\n")
print(needy.head(20).to_string(index=False))

# Bar chart of bottom-10 by GDP per capita
bottom10 = needy.nsmallest(10, 'gdpp')
plt.figure(figsize=(12, 5))
plt.barh(bottom10['country'], bottom10['gdpp'], color='#e74c3c', edgecolor='white')
plt.xlabel('GDP per Capita (USD)', fontsize=12)
plt.title('Top 10 Countries Needing Aid — Lowest GDP per Capita', fontsize=14)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()